# Case File 02: The Suspicious Spreadsheet

## Will They Show Up? Campus Event Predictor

The Campus Events Intelligence Unit has received 320 historical event records. Unfortunately, the spreadsheet appears to have been maintained by several people, one hurried robot, and possibly a sandwich.

Your job is to make the evidence trustworthy enough for Module 7. This is a **synthetic dataset**. It describes fictional events and contains no real student information.

**Routine:** Predict, inspect, decide, clean, verify, explain. Never hide a cleaning decision.

## 0. Import the case files into Colab

1. Download `campus_events_raw.csv` and this notebook from Canvas.
2. Open [Google Colab](https://colab.research.google.com/) and upload the notebook.
3. Use the folder icon to upload the CSV into the current session.
4. Confirm the filename before running the next cell.
5. Save a copy of the notebook in Google Drive. Colab session uploads can disappear when the runtime resets.

If you keep the CSV in Drive instead, mount Drive and update `DATA_PATH`. Start with the direct upload method because it has fewer moving parts.

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&logo=youtube&logoColor=white"></a> <a href="https://chatgpt.com/?q=Explain+what+this+Python+code+does%2C+step+by+step%2C+in+simple+beginner-friendly+terms.+Do+not+just+repeat+the+code+back+to+me%3B+explain+the+%2Awhy%2A+behind+each+line.%0A%0A%23+pandas+gives+Python+tools+for+working+with+tables.%0Aimport+pandas+as+pd%0Afrom+pathlib+import+Path%0A%0ADATA_PATH+%3D+Path%28%27campus_events_raw.csv%27%29%0A%0A%23+Stop+early+with+a+useful+message+if+the+file+is+missing.%0Aif+not+DATA_PATH.exists%28%29%3A%0A++++raise+FileNotFoundError%28%0A++++++++%22Upload+campus_events_raw.csv+using+Colab%27s+folder+panel%2C+then+run+this+cell+again.%22%0A++++%29%0A%0Aevents+%3D+pd.read_csv%28DATA_PATH%29%0Aprint%28%27Case+file+loaded%3A%27%2C+DATA_PATH.name%29%0Aevents.head%28%29"><img src="https://img.shields.io/badge/%F0%9F%A4%96_Explain_with_ChatGPT-10a37f?style=flat"></a></td></tr></table>

In [ ]:
# pandas gives Python tools for working with tables.
import pandas as pd
from pathlib import Path

DATA_PATH = Path('campus_events_raw.csv')

# Stop early with a useful message if the file is missing.
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Upload campus_events_raw.csv using Colab's folder panel, then run this cell again."
    )

events = pd.read_csv(DATA_PATH)
print('Case file loaded:', DATA_PATH.name)
events.head()

## 1. Predict before inspecting

Before running the health check, write three problems you think a historical event spreadsheet might contain. Prediction makes inspection purposeful.

1. I predict...
2. I predict...
3. I predict...

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&logo=youtube&logoColor=white"></a> <a href="https://chatgpt.com/?q=Explain+what+this+Python+code+does%2C+step+by+step%2C+in+simple+beginner-friendly+terms.+Do+not+just+repeat+the+code+back+to+me%3B+explain+the+%2Awhy%2A+behind+each+line.%0A%0A%23+Shape+tells+us+the+number+of+rows+and+columns.%0Aprint%28%27Rows+and+columns%3A%27%2C+events.shape%29%0Aprint%28%27%5CnColumns%3A%27%2C+events.columns.tolist%28%29%29%0Aprint%28%27%5CnData+types%3A%27%29%0Aprint%28events.dtypes%29%0Aprint%28%27%5CnMissing+values%3A%27%29%0Aprint%28events.isna%28%29.sum%28%29%29%0Aprint%28%27%5CnExact+duplicate+rows%3A%27%2C+events.duplicated%28%29.sum%28%29%29%0Aprint%28%27Repeated+event+IDs%3A%27%2C+events.duplicated%28subset%3D%5B%27event_id%27%5D%29.sum%28%29%29"><img src="https://img.shields.io/badge/%F0%9F%A4%96_Explain_with_ChatGPT-10a37f?style=flat"></a></td></tr></table>

In [ ]:
# Shape tells us the number of rows and columns.
print('Rows and columns:', events.shape)
print('\nColumns:', events.columns.tolist())
print('\nData types:')
print(events.dtypes)
print('\nMissing values:')
print(events.isna().sum())
print('\nExact duplicate rows:', events.duplicated().sum())
print('Repeated event IDs:', events.duplicated(subset=['event_id']).sum())

## 2. Understand the source before changing values

**Data Source Passport**

- Creator: course team
- Purpose: beginner data-cleaning and machine-learning practice
- Unit of one row: one fictional campus event
- Collection method: generated from transparent classroom rules with controlled randomness
- Privacy: no real people, schools, or events
- License and use: course practice material
- Known limitation: simplified patterns do not represent every reason people attend events

Synthetic does not mean perfect or neutral. The design choices still shape what the later model can learn.

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&logo=youtube&logoColor=white"></a> <a href="https://chatgpt.com/?q=Explain+what+this+Python+code+does%2C+step+by+step%2C+in+simple+beginner-friendly+terms.+Do+not+just+repeat+the+code+back+to+me%3B+explain+the+%2Awhy%2A+behind+each+line.%0A%0A%23+Scan+categories+exactly+as+written.+Whitespace+and+capitalization+matter.%0Acategory_columns+%3D+%5B%0A++++%27event_type%27%2C+%27organizer_type%27%2C+%27day_of_week%27%2C+%27venue_type%27%2C%0A++++%27indoor_outdoor%27%2C+%27registration_required%27%2C+%27free_event%27%2C%0A++++%27food_available%27%2C+%27food_type%27%2C+%27prize_available%27%2C+%27email_campaign%27%2C%0A++++%27weather_forecast%27%2C+%27actual_weather%27%2C+%27campus_activity_level%27%2C+%27high_turnout%27%0A%5D%0Afor+column+in+category_columns%3A%0A++++print%28f%27%5Cn%7Bcolumn%7D%3A%27%29%0A++++print%28events%5Bcolumn%5D.value_counts%28dropna%3DFalse%29%29%0A%0A%23+Convert+temporary+copies+to+numbers+so+suspicious+text+becomes+visible+as+missing.%0Anumeric_columns+%3D+%5B%0A++++%27start_hour%27%2C+%27duration_hours%27%2C+%27capacity%27%2C+%27accessibility_score%27%2C%0A++++%27ticket_price_usd%27%2C+%27promotion_days%27%2C+%27promotion_channels%27%2C+%27social_posts%27%2C%0A++++%27poster_count%27%2C+%27competing_events%27%2C+%27previous_similar_attendance%27%2C%0A++++%27actual_attendance%27%2C+%27attendance_rate%27%0A%5D%0Afor+column+in+numeric_columns%3A%0A++++numeric_preview+%3D+pd.to_numeric%28events%5Bcolumn%5D%2C+errors%3D%27coerce%27%29%0A++++print%28column%2C+%27values+that+are+missing+or+not+numeric%3A%27%2C+numeric_preview.isna%28%29.sum%28%29%29"><img src="https://img.shields.io/badge/%F0%9F%A4%96_Explain_with_ChatGPT-10a37f?style=flat"></a></td></tr></table>

In [ ]:
# Scan categories exactly as written. Whitespace and capitalization matter.
category_columns = [
    'event_type', 'organizer_type', 'day_of_week', 'venue_type',
    'indoor_outdoor', 'registration_required', 'free_event',
    'food_available', 'food_type', 'prize_available', 'email_campaign',
    'weather_forecast', 'actual_weather', 'campus_activity_level', 'high_turnout'
]
for column in category_columns:
    print(f'\n{column}:')
    print(events[column].value_counts(dropna=False))

# Convert temporary copies to numbers so suspicious text becomes visible as missing.
numeric_columns = [
    'start_hour', 'duration_hours', 'capacity', 'accessibility_score',
    'ticket_price_usd', 'promotion_days', 'promotion_channels', 'social_posts',
    'poster_count', 'competing_events', 'previous_similar_attendance',
    'actual_attendance', 'attendance_rate'
]
for column in numeric_columns:
    numeric_preview = pd.to_numeric(events[column], errors='coerce')
    print(column, 'values that are missing or not numeric:', numeric_preview.isna().sum())

## 3. Create a decision log before cleaning

For each issue, decide whether to correct, remove, replace, or flag it. A decision is only defensible when the rule is stated. The supplied rules below are appropriate for this synthetic practice file, not universal rules for every dataset.

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&logo=youtube&logoColor=white"></a> <a href="https://chatgpt.com/?q=Explain+what+this+Python+code+does%2C+step+by+step%2C+in+simple+beginner-friendly+terms.+Do+not+just+repeat+the+code+back+to+me%3B+explain+the+%2Awhy%2A+behind+each+line.%0A%0A%23+Preserve+the+original+evidence+and+work+on+a+copy.%0Aclean+%3D+events.copy%28%29%0A%0A%23+Remove+accidental+spaces+and+standardize+capitalization.%0Acategory_columns+%3D+%5B%0A++++%27event_type%27%2C+%27organizer_type%27%2C+%27day_of_week%27%2C+%27venue_type%27%2C%0A++++%27indoor_outdoor%27%2C+%27registration_required%27%2C+%27free_event%27%2C+%27food_available%27%2C%0A++++%27food_type%27%2C+%27prize_available%27%2C+%27email_campaign%27%2C+%27weather_forecast%27%2C%0A++++%27actual_weather%27%2C+%27campus_activity_level%27%2C+%27high_turnout%27%0A%5D%0Afor+column+in+category_columns%3A%0A++++clean%5Bcolumn%5D+%3D+clean%5Bcolumn%5D.astype%28str%29.str.strip%28%29.str.lower%28%29%0A%0A%23+Standardize+only+variations+whose+meanings+are+known.%0Aclean%5B%27day_of_week%27%5D+%3D+clean%5B%27day_of_week%27%5D.replace%28%7B%27fri%27%3A+%27friday%27%7D%29%0Aclean%5B%27actual_weather%27%5D+%3D+clean%5B%27actual_weather%27%5D.replace%28%7B%27rainy%27%3A+%27rain%27%7D%29%0Afor+column+in+%5B%27registration_required%27%2C+%27free_event%27%2C+%27food_available%27%2C%0A+++++++++++++++%27prize_available%27%2C+%27email_campaign%27%2C+%27high_turnout%27%5D%3A%0A++++clean%5Bcolumn%5D+%3D+clean%5Bcolumn%5D.replace%28%0A++++++++%7B%27y%27%3A+%27yes%27%2C+%27n%27%3A+%27no%27%2C+%27true%27%3A+%27yes%27%2C+%27false%27%3A+%27no%27%7D%0A++++%29%0A%0A%23+Convert+readable+dates+to+one+consistent+format.%0Aclean%5B%27event_date%27%5D+%3D+pd.to_datetime%28clean%5B%27event_date%27%5D%2C+errors%3D%27coerce%27%29.dt.strftime%28%27%25Y-%25m-%25d%27%29%0A%0A%23+Convert+a+known+number+word.+Unknown+words+should+not+be+guessed.%0Aclean%5B%27promotion_channels%27%5D+%3D+clean%5B%27promotion_channels%27%5D.replace%28%7B%27three%27%3A+3%7D%29%0A%0A%23+Remove+a+currency+symbol%2C+then+turn+invalid+text+such+as+%27sandwich%27+into+NaN.%0Aclean%5B%27ticket_price_usd%27%5D+%3D+clean%5B%27ticket_price_usd%27%5D.astype%28str%29.str.replace%28%27%24%27%2C+%27%27%2C+regex%3DFalse%29%0Anumeric_columns+%3D+%5B%0A++++%27start_hour%27%2C+%27duration_hours%27%2C+%27capacity%27%2C+%27accessibility_score%27%2C%0A++++%27ticket_price_usd%27%2C+%27promotion_days%27%2C+%27promotion_channels%27%2C+%27social_posts%27%2C%0A++++%27poster_count%27%2C+%27competing_events%27%2C+%27previous_similar_attendance%27%2C%0A++++%27actual_attendance%27%2C+%27attendance_rate%27%0A%5D%0Afor+column+in+numeric_columns%3A%0A++++clean%5Bcolumn%5D+%3D+pd.to_numeric%28clean%5Bcolumn%5D%2C+errors%3D%27coerce%27%29%0A%0A%23+Remove+only+exact+repeated+records.+Similar+events+are+not+automatically+duplicates.%0Aduplicate_count+%3D+int%28clean.duplicated%28%29.sum%28%29%29%0Aclean+%3D+clean.drop_duplicates%28%29.copy%28%29%0A%0A%23+Flag+values+that+violate+the+documented+rules.%0Ainvalid_duration_count+%3D+int%28%28~clean%5B%27duration_hours%27%5D.between%280.5%2C+12%29%29.sum%28%29%29%0Aclean.loc%5B~clean%5B%27duration_hours%27%5D.between%280.5%2C+12%29%2C+%27duration_hours%27%5D+%3D+pd.NA%0Aclean.loc%5Bclean%5B%27ticket_price_usd%27%5D+%3C+0%2C+%27ticket_price_usd%27%5D+%3D+pd.NA%0Aclean.loc%5B~clean%5B%27accessibility_score%27%5D.between%281%2C+5%29%2C+%27accessibility_score%27%5D+%3D+pd.NA%0Aclean.loc%5B~clean%5B%27poster_count%27%5D.between%280%2C+200%29%2C+%27poster_count%27%5D+%3D+pd.NA%0Aclean.loc%5Bclean%5B%27actual_attendance%27%5D+%3E+clean%5B%27capacity%27%5D%2C+%27actual_attendance%27%5D+%3D+pd.NA%0A%0A%23+Use+the+median+only+for+selected+practice+columns%2C+and+record+the+limitation.%0Afor+column+in+%5B%27duration_hours%27%2C+%27ticket_price_usd%27%2C+%27promotion_days%27%2C%0A+++++++++++++++%27accessibility_score%27%2C+%27poster_count%27%5D%3A%0A++++clean%5Bcolumn%5D+%3D+clean%5Bcolumn%5D.fillna%28clean%5Bcolumn%5D.median%28%29%29%0A%0A%23+Resolve+contradictions+using+the+definitions+in+the+data+dictionary.%0Aclean.loc%5Bclean%5B%27free_event%27%5D+%3D%3D+%27yes%27%2C+%27ticket_price_usd%27%5D+%3D+0%0Aclean.loc%5Bclean%5B%27food_available%27%5D+%3D%3D+%27no%27%2C+%27food_type%27%5D+%3D+%27none%27%0A%0A%23+Estimate+missing+or+impossible+attendance+with+a+transparent+practice+rule.%0Avalid_rates+%3D+clean%5B%27attendance_rate%27%5D.where%28clean%5B%27attendance_rate%27%5D.between%280%2C+1%29%29%0Amedian_rate+%3D+valid_rates.median%28%29%0Amissing_attendance+%3D+clean%5B%27actual_attendance%27%5D.isna%28%29%0Aclean.loc%5Bmissing_attendance%2C+%27actual_attendance%27%5D+%3D+%28%0A++++clean.loc%5Bmissing_attendance%2C+%27capacity%27%5D+%2A+median_rate%0A%29.round%28%29%0A%0A%23+Recalculate+derived+outcomes+so+they+agree+with+the+cleaned+evidence.%0Aclean%5B%27attendance_rate%27%5D+%3D+%28clean%5B%27actual_attendance%27%5D+%2F+clean%5B%27capacity%27%5D%29.round%283%29%0Aclean%5B%27high_turnout%27%5D+%3D+clean%5B%27attendance_rate%27%5D.ge%280.65%29.map%28%7BTrue%3A+%27yes%27%2C+False%3A+%27no%27%7D%29%0A%0Acleaning_log+%3D+pd.DataFrame%28%5B%0A++++%7B%27issue%27%3A+%27spacing%2C+capitalization%2C+and+dates%27%2C+%27decision%27%3A+%27standardized+known+formats%27%2C+%27risk%27%3A+%27unknown+meanings+must+not+be+guessed%27%7D%2C%0A++++%7B%27issue%27%3A+%27numeric+text+and+range+errors%27%2C+%27decision%27%3A+%27converted+types+and+flagged+invalid+values%27%2C+%27risk%27%3A+%27a+rule+cannot+confirm+what+originally+happened%27%7D%2C%0A++++%7B%27issue%27%3A+%27exact+duplicates%27%2C+%27decision%27%3A+f%27removed+%7Bduplicate_count%7D+repeated+records%27%2C+%27risk%27%3A+%27similar+events+are+not+automatically+duplicates%27%7D%2C%0A++++%7B%27issue%27%3A+%27implausible+duration%27%2C+%27decision%27%3A+f%27flagged+%7Binvalid_duration_count%7D+value%3B+median+filled%27%2C+%27risk%27%3A+%27real+projects+require+source+confirmation%27%7D%2C%0A++++%7B%27issue%27%3A+%27free-event+and+food+contradictions%27%2C+%27decision%27%3A+%27applied+data-dictionary+definitions%27%2C+%27risk%27%3A+%27the+original+entry+may+still+need+investigation%27%7D%2C%0A++++%7B%27issue%27%3A+%27missing+or+impossible+attendance%27%2C+%27decision%27%3A+%27estimated+from+median+valid+attendance+rate%27%2C+%27risk%27%3A+%27estimated+outcomes+add+uncertainty%27%7D%2C%0A++++%7B%27issue%27%3A+%27derived+outcomes%27%2C+%27decision%27%3A+%27recalculated+attendance_rate+and+high_turnout%27%2C+%27risk%27%3A+%27the+65+percent+threshold+is+a+course+design+choice%27%7D%2C%0A%5D%29%0Acleaning_log"><img src="https://img.shields.io/badge/%F0%9F%A4%96_Explain_with_ChatGPT-10a37f?style=flat"></a></td></tr></table>

In [ ]:
# Preserve the original evidence and work on a copy.
clean = events.copy()

# Remove accidental spaces and standardize capitalization.
category_columns = [
    'event_type', 'organizer_type', 'day_of_week', 'venue_type',
    'indoor_outdoor', 'registration_required', 'free_event', 'food_available',
    'food_type', 'prize_available', 'email_campaign', 'weather_forecast',
    'actual_weather', 'campus_activity_level', 'high_turnout'
]
for column in category_columns:
    clean[column] = clean[column].astype(str).str.strip().str.lower()

# Standardize only variations whose meanings are known.
clean['day_of_week'] = clean['day_of_week'].replace({'fri': 'friday'})
clean['actual_weather'] = clean['actual_weather'].replace({'rainy': 'rain'})
for column in ['registration_required', 'free_event', 'food_available',
               'prize_available', 'email_campaign', 'high_turnout']:
    clean[column] = clean[column].replace(
        {'y': 'yes', 'n': 'no', 'true': 'yes', 'false': 'no'}
    )

# Convert readable dates to one consistent format.
clean['event_date'] = pd.to_datetime(clean['event_date'], errors='coerce').dt.strftime('%Y-%m-%d')

# Convert a known number word. Unknown words should not be guessed.
clean['promotion_channels'] = clean['promotion_channels'].replace({'three': 3})

# Remove a currency symbol, then turn invalid text such as 'sandwich' into NaN.
clean['ticket_price_usd'] = clean['ticket_price_usd'].astype(str).str.replace('$', '', regex=False)
numeric_columns = [
    'start_hour', 'duration_hours', 'capacity', 'accessibility_score',
    'ticket_price_usd', 'promotion_days', 'promotion_channels', 'social_posts',
    'poster_count', 'competing_events', 'previous_similar_attendance',
    'actual_attendance', 'attendance_rate'
]
for column in numeric_columns:
    clean[column] = pd.to_numeric(clean[column], errors='coerce')

# Remove only exact repeated records. Similar events are not automatically duplicates.
duplicate_count = int(clean.duplicated().sum())
clean = clean.drop_duplicates().copy()

# Flag values that violate the documented rules.
invalid_duration_count = int((~clean['duration_hours'].between(0.5, 12)).sum())
clean.loc[~clean['duration_hours'].between(0.5, 12), 'duration_hours'] = pd.NA
clean.loc[clean['ticket_price_usd'] < 0, 'ticket_price_usd'] = pd.NA
clean.loc[~clean['accessibility_score'].between(1, 5), 'accessibility_score'] = pd.NA
clean.loc[~clean['poster_count'].between(0, 200), 'poster_count'] = pd.NA
clean.loc[clean['actual_attendance'] > clean['capacity'], 'actual_attendance'] = pd.NA

# Use the median only for selected practice columns, and record the limitation.
for column in ['duration_hours', 'ticket_price_usd', 'promotion_days',
               'accessibility_score', 'poster_count']:
    clean[column] = clean[column].fillna(clean[column].median())

# Resolve contradictions using the definitions in the data dictionary.
clean.loc[clean['free_event'] == 'yes', 'ticket_price_usd'] = 0
clean.loc[clean['food_available'] == 'no', 'food_type'] = 'none'

# Estimate missing or impossible attendance with a transparent practice rule.
valid_rates = clean['attendance_rate'].where(clean['attendance_rate'].between(0, 1))
median_rate = valid_rates.median()
missing_attendance = clean['actual_attendance'].isna()
clean.loc[missing_attendance, 'actual_attendance'] = (
    clean.loc[missing_attendance, 'capacity'] * median_rate
).round()

# Recalculate derived outcomes so they agree with the cleaned evidence.
clean['attendance_rate'] = (clean['actual_attendance'] / clean['capacity']).round(3)
clean['high_turnout'] = clean['attendance_rate'].ge(0.65).map({True: 'yes', False: 'no'})

cleaning_log = pd.DataFrame([
    {'issue': 'spacing, capitalization, and dates', 'decision': 'standardized known formats', 'risk': 'unknown meanings must not be guessed'},
    {'issue': 'numeric text and range errors', 'decision': 'converted types and flagged invalid values', 'risk': 'a rule cannot confirm what originally happened'},
    {'issue': 'exact duplicates', 'decision': f'removed {duplicate_count} repeated records', 'risk': 'similar events are not automatically duplicates'},
    {'issue': 'implausible duration', 'decision': f'flagged {invalid_duration_count} value; median filled', 'risk': 'real projects require source confirmation'},
    {'issue': 'free-event and food contradictions', 'decision': 'applied data-dictionary definitions', 'risk': 'the original entry may still need investigation'},
    {'issue': 'missing or impossible attendance', 'decision': 'estimated from median valid attendance rate', 'risk': 'estimated outcomes add uncertainty'},
    {'issue': 'derived outcomes', 'decision': 'recalculated attendance_rate and high_turnout', 'risk': 'the 65 percent threshold is a course design choice'},
])
cleaning_log

## 4. Filter, sort, and summarize without claiming cause

Filtering keeps selected rows. Sorting changes their order. Grouping summarizes many rows. These operations can reveal a pattern, but they cannot prove why it occurred.

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&logo=youtube&logoColor=white"></a> <a href="https://chatgpt.com/?q=Explain+what+this+Python+code+does%2C+step+by+step%2C+in+simple+beginner-friendly+terms.+Do+not+just+repeat+the+code+back+to+me%3B+explain+the+%2Awhy%2A+behind+each+line.%0A%0A%23+Focus+on+rainy+outdoor+events+and+show+the+least+attended+first.%0Afocused_events+%3D+clean%5B%0A++++%28clean%5B%27indoor_outdoor%27%5D+%3D%3D+%27outdoor%27%29+%26+%28clean%5B%27actual_weather%27%5D+%3D%3D+%27rain%27%29%0A%5D.sort_values%28%27actual_attendance%27%29%0Afocused_events%5B%5B%27event_id%27%2C+%27event_type%27%2C+%27actual_weather%27%2C+%27actual_attendance%27%2C+%27attendance_rate%27%5D%5D.head%2810%29%0A%0A%23+Compare+turnout+by+food+availability.+This+is+descriptive%2C+not+causal.%0Afood_summary+%3D+clean.groupby%28%27food_available%27%29.agg%28%0A++++events%3D%28%27event_id%27%2C+%27count%27%29%2C%0A++++average_attendance%3D%28%27actual_attendance%27%2C+%27mean%27%29%2C%0A++++median_attendance%3D%28%27actual_attendance%27%2C+%27median%27%29%2C%0A%29.round%281%29%0Afood_summary"><img src="https://img.shields.io/badge/%F0%9F%A4%96_Explain_with_ChatGPT-10a37f?style=flat"></a></td></tr></table>

In [ ]:
# Focus on rainy outdoor events and show the least attended first.
focused_events = clean[
    (clean['indoor_outdoor'] == 'outdoor') & (clean['actual_weather'] == 'rain')
].sort_values('actual_attendance')
focused_events[['event_id', 'event_type', 'actual_weather', 'actual_attendance', 'attendance_rate']].head(10)

# Compare turnout by food availability. This is descriptive, not causal.
food_summary = clean.groupby('food_available').agg(
    events=('event_id', 'count'),
    average_attendance=('actual_attendance', 'mean'),
    median_attendance=('actual_attendance', 'median'),
).round(1)
food_summary

## 5. Compare before and after

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&logo=youtube&logoColor=white"></a> <a href="https://chatgpt.com/?q=Explain+what+this+Python+code+does%2C+step+by+step%2C+in+simple+beginner-friendly+terms.+Do+not+just+repeat+the+code+back+to+me%3B+explain+the+%2Awhy%2A+behind+each+line.%0A%0Acomparison+%3D+pd.DataFrame%28%7B%0A++++%27measure%27%3A+%5B%27rows%27%2C+%27missing+cells%27%2C+%27exact+duplicates%27%5D%2C%0A++++%27raw%27%3A+%5Blen%28events%29%2C+int%28events.isna%28%29.sum%28%29.sum%28%29%29%2C+int%28events.duplicated%28%29.sum%28%29%29%5D%2C%0A++++%27clean%27%3A+%5Blen%28clean%29%2C+int%28clean.isna%28%29.sum%28%29.sum%28%29%29%2C+int%28clean.duplicated%28%29.sum%28%29%29%5D%2C%0A%7D%29%0Acomparison"><img src="https://img.shields.io/badge/%F0%9F%A4%96_Explain_with_ChatGPT-10a37f?style=flat"></a></td></tr></table>

In [ ]:
comparison = pd.DataFrame({
    'measure': ['rows', 'missing cells', 'exact duplicates'],
    'raw': [len(events), int(events.isna().sum().sum()), int(events.duplicated().sum())],
    'clean': [len(clean), int(clean.isna().sum().sum()), int(clean.duplicated().sum())],
})
comparison

## 6. Validate the cleaned evidence

Passing these checks means the table follows our stated rules. It does not prove that every historical record is true or that the dataset represents every campus.

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&logo=youtube&logoColor=white"></a> <a href="https://chatgpt.com/?q=Explain+what+this+Python+code+does%2C+step+by+step%2C+in+simple+beginner-friendly+terms.+Do+not+just+repeat+the+code+back+to+me%3B+explain+the+%2Awhy%2A+behind+each+line.%0A%0Avalidation_checks+%3D+%7B%0A++++%27event_id_unique%27%3A+clean%5B%27event_id%27%5D.is_unique%2C%0A++++%27duration_between_0.5_and_12%27%3A+clean%5B%27duration_hours%27%5D.between%280.5%2C+12%29.all%28%29%2C%0A++++%27price_not_negative%27%3A+%28clean%5B%27ticket_price_usd%27%5D+%3E%3D+0%29.all%28%29%2C%0A++++%27free_events_cost_zero%27%3A+clean.loc%5Bclean%5B%27free_event%27%5D+%3D%3D+%27yes%27%2C+%27ticket_price_usd%27%5D.eq%280%29.all%28%29%2C%0A++++%27accessibility_between_1_and_5%27%3A+clean%5B%27accessibility_score%27%5D.between%281%2C+5%29.all%28%29%2C%0A++++%27attendance_not_negative%27%3A+%28clean%5B%27actual_attendance%27%5D+%3E%3D+0%29.all%28%29%2C%0A++++%27attendance_not_above_capacity%27%3A+%28clean%5B%27actual_attendance%27%5D+%3C%3D+clean%5B%27capacity%27%5D%29.all%28%29%2C%0A++++%27rate_matches_attendance%27%3A+%28%28clean%5B%27actual_attendance%27%5D+%2F+clean%5B%27capacity%27%5D%29.round%283%29+%3D%3D+clean%5B%27attendance_rate%27%5D%29.all%28%29%2C%0A++++%27target_matches_65_percent_rule%27%3A+%28clean%5B%27attendance_rate%27%5D.ge%280.65%29.map%28%7BTrue%3A+%27yes%27%2C+False%3A+%27no%27%7D%29+%3D%3D+clean%5B%27high_turnout%27%5D%29.all%28%29%2C%0A++++%27yes_no_columns_known%27%3A+all%28clean%5Bc%5D.isin%28%5B%27yes%27%2C+%27no%27%5D%29.all%28%29+for+c+in+%5B%27registration_required%27%2C+%27free_event%27%2C+%27food_available%27%2C+%27prize_available%27%2C+%27email_campaign%27%2C+%27high_turnout%27%5D%29%2C%0A++++%27weather_known%27%3A+clean%5B%27actual_weather%27%5D.isin%28%5B%27clear%27%2C+%27cloudy%27%2C+%27rain%27%2C+%27windy%27%5D%29.all%28%29%2C%0A%7D%0Avalidation_checks"><img src="https://img.shields.io/badge/%F0%9F%A4%96_Explain_with_ChatGPT-10a37f?style=flat"></a></td></tr></table>

In [ ]:
validation_checks = {
    'event_id_unique': clean['event_id'].is_unique,
    'duration_between_0.5_and_12': clean['duration_hours'].between(0.5, 12).all(),
    'price_not_negative': (clean['ticket_price_usd'] >= 0).all(),
    'free_events_cost_zero': clean.loc[clean['free_event'] == 'yes', 'ticket_price_usd'].eq(0).all(),
    'accessibility_between_1_and_5': clean['accessibility_score'].between(1, 5).all(),
    'attendance_not_negative': (clean['actual_attendance'] >= 0).all(),
    'attendance_not_above_capacity': (clean['actual_attendance'] <= clean['capacity']).all(),
    'rate_matches_attendance': ((clean['actual_attendance'] / clean['capacity']).round(3) == clean['attendance_rate']).all(),
    'target_matches_65_percent_rule': (clean['attendance_rate'].ge(0.65).map({True: 'yes', False: 'no'}) == clean['high_turnout']).all(),
    'yes_no_columns_known': all(clean[c].isin(['yes', 'no']).all() for c in ['registration_required', 'free_event', 'food_available', 'prize_available', 'email_campaign', 'high_turnout']),
    'weather_known': clean['actual_weather'].isin(['clear', 'cloudy', 'rain', 'windy']).all(),
}
validation_checks

## 7. Protect Module 8 from target leakage

`high_turnout` is the target. `actual_attendance` and `attendance_rate` directly determine that target, so using either as an input would give the future model the answer. `actual_weather` is also unavailable if the prediction is made before the event. Keep these columns for historical analysis, but exclude them from advance model inputs.

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&logo=youtube&logoColor=white"></a> <a href="https://chatgpt.com/?q=Explain+what+this+Python+code+does%2C+step+by+step%2C+in+simple+beginner-friendly+terms.+Do+not+just+repeat+the+code+back+to+me%3B+explain+the+%2Awhy%2A+behind+each+line.%0A%0Afuture_features+%3D+%5B%0A++++%27event_type%27%2C+%27organizer_type%27%2C+%27day_of_week%27%2C+%27start_hour%27%2C%0A++++%27duration_hours%27%2C+%27venue_type%27%2C+%27indoor_outdoor%27%2C+%27capacity%27%2C%0A++++%27registration_required%27%2C+%27accessibility_score%27%2C+%27free_event%27%2C%0A++++%27ticket_price_usd%27%2C+%27food_available%27%2C+%27food_type%27%2C+%27prize_available%27%2C%0A++++%27promotion_days%27%2C+%27promotion_channels%27%2C+%27social_posts%27%2C+%27email_campaign%27%2C%0A++++%27poster_count%27%2C+%27weather_forecast%27%2C+%27competing_events%27%2C%0A++++%27campus_activity_level%27%2C+%27previous_similar_attendance%27%0A%5D%0Atarget+%3D+%27high_turnout%27%0Aexcluded_from_model+%3D+%5B%27event_id%27%2C+%27event_date%27%2C+%27actual_weather%27%2C+%27actual_attendance%27%2C+%27attendance_rate%27%5D%0Aprint%28%27Future+features%3A%27%2C+future_features%29%0Aprint%28%27Target%3A%27%2C+target%29%0Aprint%28%27Excluded%3A%27%2C+excluded_from_model%29"><img src="https://img.shields.io/badge/%F0%9F%A4%96_Explain_with_ChatGPT-10a37f?style=flat"></a></td></tr></table>

In [ ]:
future_features = [
    'event_type', 'organizer_type', 'day_of_week', 'start_hour',
    'duration_hours', 'venue_type', 'indoor_outdoor', 'capacity',
    'registration_required', 'accessibility_score', 'free_event',
    'ticket_price_usd', 'food_available', 'food_type', 'prize_available',
    'promotion_days', 'promotion_channels', 'social_posts', 'email_campaign',
    'poster_count', 'weather_forecast', 'competing_events',
    'campus_activity_level', 'previous_similar_attendance'
]
target = 'high_turnout'
excluded_from_model = ['event_id', 'event_date', 'actual_weather', 'actual_attendance', 'attendance_rate']
print('Future features:', future_features)
print('Target:', target)
print('Excluded:', excluded_from_model)

## 8. Export the clean handoff file

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&logo=youtube&logoColor=white"></a> <a href="https://chatgpt.com/?q=Explain+what+this+Python+code+does%2C+step+by+step%2C+in+simple+beginner-friendly+terms.+Do+not+just+repeat+the+code+back+to+me%3B+explain+the+%2Awhy%2A+behind+each+line.%0A%0AOUTPUT_PATH+%3D+%27campus_events_clean.csv%27%0Aclean.to_csv%28OUTPUT_PATH%2C+index%3DFalse%29%0Aprint%28%27Saved%27%2C+OUTPUT_PATH%2C+%27with+shape%27%2C+clean.shape%29"><img src="https://img.shields.io/badge/%F0%9F%A4%96_Explain_with_ChatGPT-10a37f?style=flat"></a></td></tr></table>

In [ ]:
OUTPUT_PATH = 'campus_events_clean.csv'
clean.to_csv(OUTPUT_PATH, index=False)
print('Saved', OUTPUT_PATH, 'with shape', clean.shape)

## 9. Write the evidence note

Complete these statements using your outputs:

- The raw file contained ___ rows and ___ columns.
- The most important quality problems were ___.
- I handled them by ___.
- One descriptive pattern I observed was ___.
- This pattern does not prove ___.
- One limitation cleaning did not solve is ___.

## Required Gemini verification record

Ask Gemini to explain or review one cleaning step. Do not ask it to complete the entire case file.

- My prompt:
- Gemini suggested:
- I tested it by:
- Official pandas documentation I checked:
- I accepted, changed, or rejected the suggestion because:

Never enter private or real student information.

## Module 7 handoff

Keep these together in your Phase 2 project folder:

1. This completed notebook with outputs and explanations
2. `campus_events_clean.csv`
3. The cleaning log and evidence note inside the notebook
4. `campus_events_data_dictionary.csv`

In Module 7, the investigation becomes **Patterns in the Crowd**. You will use statistics and charts to ask what successful events appear to share, while avoiding unsupported causal claims.